# 05 — Add a New Stock


## 1 — Bootstrap


In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
SRC = ROOT / 'src'
assert SRC.exists(), f'Could not find src/ at {SRC}'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

import config
from collect import (collect_prices, registry_template, verify_all_raw,
                     verify_raw_file)
from pipeline import TrainConfig, artifacts_exist, predict_latest, train_stock
from validate import run_all_checks

config.ensure_dirs()
pd.set_option('display.width', 140)

print(f'Registered stocks: {config.list_stocks()}')


Registered stocks: ['RELIANCE', 'TCS']


## 2 — Health check


In [2]:
health = run_all_checks(deep=False)


  PROJECT HEALTH CHECK

Dependencies
------------
  [ok] import numpy             core
  [ok] import pandas            core
  [ok] import sklearn           core
  [ok] import joblib            core
  [ok] import ta                indicators
  [ok] import matplotlib        plots
  [ok] import yfinance          downloading new price data
  [ok] import torch             LSTM / GRU sequence models
  [ok] import streamlit         the dashboard
  [ok] import plotly            the dashboard
  [ok] import transformers      FinBERT sentiment
  [ok] import vaderSentiment    VADER sentiment
  [ok] import requests          news collection

Project modules
---------------
  [ok] src/config.py            imports cleanly
  [ok] src/dataio.py            imports cleanly
  [ok] src/features.py          imports cleanly
  [ok] src/dataset.py           imports cleanly
  [ok] src/stationary.py        imports cleanly
  [ok] src/targets.py           imports cleanly
  [ok] src/evaluation.py        imports clea

KeyboardInterrupt: 

## 3 — Define the stock to add


In [3]:
NEW_KEY = 'INFY'
NEW_TICKER = 'INFY.NS'
NEW_NAME = 'Infosys'
NEW_START = '2014-01-01'
NEW_END = '2024-01-01'

print(registry_template(NEW_KEY, NEW_TICKER, NEW_NAME,
                        start=NEW_START, end=NEW_END))


    "INFY": StockConfig(
        key="INFY",
        ticker="INFY.NS",
        display_name="Infosys",
        raw_filename="INFY_raw.csv",
        start="2014-01-01",
        end="2024-01-01",
        sentiment_filename=None,
        notes="Added 2026-08-30.",
    ),


## 4 — Paste the block above into `src/config.py`, then restart the kernel


In [ ]:
import importlib

import config
importlib.reload(config)

print(f'Registered stocks: {config.list_stocks()}')
assert NEW_KEY in config.STOCKS, (
    f'{NEW_KEY} is not in config.STOCKS. Paste the block from step 3 into '
    f'src/config.py, save the file, then restart the kernel and re-run.')
print(f'{NEW_KEY} found in registry')


Registered stocks: ['INFY', 'RELIANCE', 'TCS']
INFY found in registry


## 5 — Download price history


In [5]:
path = collect_prices(NEW_KEY, overwrite=False)
print()
result = verify_raw_file(NEW_KEY)


  INFY: downloading INFY.NS 2014-01-01 .. 2024-01-01
  INFY: saved 2465 rows -> data/raw/INFY_raw.csv

  INFY         OK  2465 rows, 2014-01-01 .. 2023-12-29 (10.0y)


## 6 — Confirm the history is long enough


In [6]:
if not result.get('ok'):
    raise RuntimeError(f"Download failed: {result.get('problem')}")

if result.get('warning'):
    print('WARNING')
    print(f"  {result['warning']}")
    print()
    print('  The pipeline will still train, at a shorter horizon.')
    print('  For a horizon-3 model, re-run step 5 with an earlier start date.')
else:
    print(f"{NEW_KEY}: {result['rows']} rows over {result['years']} years.")
    print('Sufficient for a horizon-3 model.')


INFY: 2465 rows over 10.0 years.
Sufficient for a horizon-3 model.


## 7 — Build the dataset


In [7]:
from dataset import build_dataset
from features import audit_dataset, print_audit
from stationary import add_stationary_features, get_stationary_features

df = build_dataset(NEW_KEY, with_sentiment=False, save=True, verbose=True)
df = add_stationary_features(df).dropna().reset_index(drop=True)
features = get_stationary_features(df)

print_audit(audit_dataset(df, features), title=f'Audit: {NEW_KEY}')



=== Building dataset: INFY (Infosys) ===
  loaded INFY_raw.csv: 2465 rows (0 dropped), 2014-01-01 to 2023-12-29
  features: 2465 -> 2415 rows (50 dropped as warm-up/unlabelled), 35 columns

Audit: INFY
-----------
  rows                 : 2415
  columns              : 35
  leaky_features       : None
  has_inf              : False
  nan_counts           : None
  dates_sorted         : True
  duplicate_dates      : 0
  date_range           : 2014-03-12 to 2023-12-28
  target_balance       : {1: 0.5222, 0: 0.4778}
  majority_baseline    : 0.5222
  sentiment_merged     : False
  n_features           : 20
  saved INFY_dataset.csv: 2415 rows x 35 cols -> processed/

Audit: INFY
-----------
  rows                 : 2354
  columns              : 77
  leaky_features       : None
  has_inf              : False
  nan_counts           : None
  dates_sorted         : True
  duplicate_dates      : 0
  date_range           : 2014-06-11 to 2023-12-28
  target_balance       : {1: 0.5225, 0: 0.4775}
 

## 8 — Train and save artifacts


In [8]:
cfg = TrainConfig()   # defaults: logistic, horizon 3, non-overlapping

artifacts = train_stock(NEW_KEY, cfg=cfg, verbose=True)



=== Training INFY (Infosys) ===
  full frame     : 2354 rows, 52 features
  modelling frame: 780 rows (horizon=3, k=0.0)
  walk-forward   : 0.5077 +/- 0.0605 (baseline 0.5615, edge -0.0538)
  seed-averaged  : 0.5077 (edge -0.0538) -> NO EDGE
  permutation    : NO SIGNAL (shuffled 0.5244)
  operating point: coverage 20%, accuracy 0.6154, edge +0.0641
  abstention     : ADOPTED (edge improves -0.0538 -> +0.0641 at 20% coverage)
  saved -> models/INFY/


## 9 — Verify the trained model


In [10]:
meta = artifacts['metadata']
perm = meta['permutation'] or {}
seeds = meta['seed_robustness'] or {}

summary = pd.Series({
    'stock': meta['stock_key'],
    'rows': meta['n_modelling_rows'],
    'features': meta['n_features'],
    'horizon': meta['config']['horizon'],
    'accuracy': meta['walk_forward_accuracy'],
    'std': meta['walk_forward_std'],
    'baseline': meta['baseline'],
    'edge': meta['edge'],
    'permutation': perm.get('verdict'),
    'seed_verdict': seeds.get('verdict'),
})
print(summary.to_string())
print()

if perm.get('verdict') == 'LEAKAGE':
    print('STOP: permutation test indicates leakage. Do not use this model.')
elif meta['edge'] <= 0:
    print('No demonstrated edge. The model is valid but has no skill on this')
    print('stock. The dashboard will show a warning banner. This is a normal')
    print('and reportable outcome, not a bug.')
else:
    print('Model has a positive, permutation-clean edge.')


stock                INFY
rows                  780
features               52
horizon                 3
accuracy           0.5077
std                0.0605
baseline           0.5615
edge              -0.0538
permutation     NO SIGNAL
seed_verdict      NO EDGE

No demonstrated edge. The model is valid but has no skill on this
stock. The dashboard will show a warning banner. This is a normal
and reportable outcome, not a bug.


## 10 — Offset robustness


In [11]:
from modeling import evaluate_across_offsets
from targets import build_directional_dataset

horizon = meta['config']['horizon']

if horizon > 1:
    def builder(frame, offset):
        return build_directional_dataset(
            frame.iloc[offset:].reset_index(drop=True),
            horizon=horizon, k=meta['config']['k'], non_overlapping=True)

    offs, edges = evaluate_across_offsets(
        df, features, meta['config']['model_name'],
        dataset_builder=builder, n_offsets=horizon,
        n_splits=meta['config']['n_splits'],
        embargo=meta['config']['embargo'])

    off_df = pd.DataFrame({'offset': offs, 'edge': [round(e, 4) for e in edges]})
    print(off_df.to_string(index=False))
    print()
    if all(e > 0 for e in edges):
        print('Positive at every sampling offset. The edge is robust.')
    else:
        print('Edge flips negative at one or more offsets.')
        print('Report the mean across offsets, not the best one.')
else:
    print('Horizon is 1; there is only one sampling offset to test.')


 offset    edge
      0 -0.0538
      1 -0.0154
      2 -0.0308

Edge flips negative at one or more offsets.
Report the mean across offsets, not the best one.


## 11 — Final check


In [12]:
print(predict_latest(NEW_KEY))
print()

rows = []
for key in config.list_stocks():
    rows.append({'stock': key, 'trained': artifacts_exist(key)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print('Launch the dashboard with:   streamlit run app.py')


{'stock': 'INFY', 'date': '2023-12-28', 'close': 1440.049560546875, 'probability_up': 0.68, 'confidence': 0.18, 'confidence_threshold': 0.211, 'signal': 'NO SIGNAL', 'horizon_days': 3, 'expected_accuracy': 0.6154, 'expected_coverage': 0.2}

   stock  trained
    INFY     True
RELIANCE     True
     TCS     True

Launch the dashboard with:   streamlit run app.py


## 12 — Rebuild the comparison report


In [13]:
import json
from pipeline import load_artifacts

rows = []
for key in config.list_stocks():
    if not artifacts_exist(key):
        continue
    m = load_artifacts(key)['metadata']
    rows.append({
        'stock': key,
        'horizon': m['config']['horizon'],
        'model': m['config']['model_name'],
        'rows': m['n_modelling_rows'],
        'accuracy': m['walk_forward_accuracy'],
        'std': m['walk_forward_std'],
        'baseline': m['baseline'],
        'edge': m['edge'],
        'permutation': (m['permutation'] or {}).get('verdict'),
    })

table = pd.DataFrame(rows).set_index('stock')
table.to_csv(config.REPORTS_DIR / 'all_stocks_summary.csv')
print(table.to_string())


          horizon     model  rows  accuracy     std  baseline    edge permutation
stock                                                                            
INFY            3  logistic   780    0.5077  0.0605    0.5615 -0.0538   NO SIGNAL
RELIANCE        1  logistic   372    0.4679  0.0685    0.5111 -0.0432   NO SIGNAL
TCS             3  logistic   780    0.5333  0.0525    0.5179  0.0154      SIGNAL
